# **Residual stream: practice**

In the lessons "Residual stream as a communication channel", "MLP as key-value memory" and "Head composition and virtual heads" everything was computed by hand on small vectors. Here the same three tools are applied to GPT-2 small from TransformerLens: 12 layers, 12 attention heads each, a residual stream of dimension 768.

In this practice you will:

* decompose the logit of the token " Paris" on the prompt "The Eiffel Tower is in the city of" into direct contributions of blocks and heads (direct logit attribution);
* find the MLP neuron that pushes " Paris" up most directly and look at which tokens its value vector promotes;
* compute from the weights the Q-, K- and V-composition scores between the heads of layers 4 and 5 ([Elhage et al., 2021](https://transformer-circuits.pub/2021/framework/index.html)).

The tasks are numbered; you enter the answers in the lesson "Residual stream: Practice".

## **Setup**
A CPU is enough: the model is small, and everything here is forward passes.

In [ ]:
!pip install transformer-lens==3.9.0 -q

In [ ]:
import torch
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)  # forward passes only
model = HookedTransformer.from_pretrained("gpt2-small", device="cpu")
print(model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_model, model.cfg.d_mlp)

## **Prompt and prediction**

Run the prompt and keep the cache of all activations. The logit of a token is its coordinate in the model output at the last position.

In [ ]:
prompt = "The Eiffel Tower is in the city of"
tokens = model.to_tokens(prompt)
print(model.to_str_tokens(prompt))
logits, cache = model.run_with_cache(tokens)
paris = model.to_single_token(" Paris")

top = logits[0, -1].topk(5)
for value, index in zip(top.values, top.indices):
    print(repr(model.to_string(index)), round(value.item(), 2))
print("logit Paris:", round(logits[0, -1, paris].item(), 2))

**Task 1. Which token does the model rank first after this prompt?**

**Task 2. What is the logit of the token " Paris"? Round to two decimals.**

## **Decomposing the logit by blocks**

The final state of the stream is the sum of the token embedding, the positional embedding and the contributions of all attention and MLP blocks:

$$h_L = h_0 + \sum_{l} (a_l + m_l).$$

The final LayerNorm stands before the unembedding. If its scale is fixed at the value from this run, it acts on every term in the same way, and the logit decomposes exactly: `cache.decompose_resid(..., apply_ln=True)` returns the terms after LayerNorm, and the direct contribution of a term is its dot product with the column of $W_U$ for " Paris". The bias $b_U$ is added to the sum of contributions.

In [ ]:
stack, labels = cache.decompose_resid(layer=-1, pos_slice=-1, apply_ln=True, return_labels=True)
w = model.W_U[:, paris]  # unembedding column of " Paris"
contrib = ...  # Your code here

print("sum + b_U:", round((contrib.sum() + model.b_U[paris]).item(), 3), " logit:", round(logits[0, -1, paris].item(), 3))
for label, value in sorted(zip(labels, contrib.tolist()), key=lambda x: -x[1]):
    print(f"{label:>12} {value:7.3f}")

**Task 3. Which block makes the largest direct contribution to the logit of " Paris"?**

**Task 4. What is this contribution? Round to two decimals.**

## **Contributions of attention heads**

The contribution of an attention block is the sum of its heads' contributions. `cache.stack_head_results(..., apply_ln=True)` returns the output of every head after $W_O$ with the same fixed LayerNorm scale.

In [ ]:
head_stack, head_labels = cache.stack_head_results(layer=-1, pos_slice=-1, apply_ln=True, return_labels=True)
head_contrib = ...  # Your code here

for label, value in sorted(zip(head_labels, head_contrib.tolist()), key=lambda x: -x[1])[:5]:
    print(f"{label:>6} {value:7.3f}")

**Task 5. Which head makes the largest direct contribution to the logit of " Paris"?** Write the head as "layer.head", for example 1.2.

**Task 6. What is the contribution of this head? Round to two decimals.**

## **MLP as key-value memory**

The output of MLP layer $l$ is the sum of value vectors weighted by activations:

$$m_l = \sum_i \sigma(k_i^\top h)\, v_i + b_{out},$$

where $\sigma(k_i^\top h)$ is the activation of neuron $i$ (in TransformerLens, `cache["post", l]`) and $v_i$ is row $i$ of `model.W_out[l]`. So the direct contribution of neuron $i$ to the logit is its activation times the dot product $\langle v_i, w \rangle$, divided by the scale $s$ of the final LayerNorm (`cache["ln_final.hook_scale"]`). Take layer 9.

In [ ]:
layer = 9
act = cache["post", layer][0, -1]
s = cache["ln_final.hook_scale"][0, -1, 0]
neuron_contrib = ...  # Your code here

block = labels.index(f"{layer}_mlp_out")
print("neurons + b_out:", round((neuron_contrib.sum() + model.b_out[layer] @ w / s).item(), 3), " block:", round(contrib[block].item(), 3))
top = neuron_contrib.topk(5)
for value, index in zip(top.values, top.indices):
    print(f"neuron {index.item():>4}  activation {act[index].item():6.3f}  contribution {value.item():6.3f}")

**Task 7. Which neuron of layer 9 makes the largest direct contribution to the logit of " Paris"? Write its index.**

**Task 8. What is the activation of this neuron? Round to two decimals.**

**Task 9. What is the direct contribution of this neuron? Round to two decimals.**

The value vector of a neuron can be projected onto the whole vocabulary at once: this shows which tokens the neuron promotes whenever it fires, on any input.

In [ ]:
neuron = ...  # Your code here
promoted = (model.W_out[layer, neuron] @ model.W_U).topk(10).indices
print([model.to_string(t) for t in promoted])

**Task 10. Which tokens does the value vector of this neuron promote?**

## **Composition from the weights**

Elhage et al. measure how much a head of a later layer reads the output of a head of an earlier layer with the Frobenius norm of a matrix product: for K-composition $\|W_{QK}^{h_2}W_{OV}^{h_1}\|_F / (\|W_{QK}^{h_2}\|_F\,\|W_{OV}^{h_1}\|_F)$, for Q- and V-composition the same with $W_{QK}^{h_2\,T}$ and $W_{OV}^{h_2}$. In TransformerLens this is `model.all_composition_scores(mode)`, a tensor `[layer 1, head 1, layer 2, head 2]`. No data is needed: the score is computed from the weights. Find the pair of heads of layers 4 and 5 with the largest K-composition score and compare it with the mean over all 144 pairs of these layers.

In [ ]:
scores = {mode: model.all_composition_scores(mode)[4, :, 5, :] for mode in "QKV"}  # [head of layer 4, head of layer 5]
h1, h2 = ...  # Your code here

print(f"pair 4.{h1} -> 5.{h2}")
for mode in "QKV":
    print(mode, round(scores[mode][h1, h2].item(), 4))
print("mean K over 144 pairs:", round(scores["K"].mean().item(), 4))

**Task 11. Between which heads of layers 4 and 5 is the K-composition score largest?**

**Task 12. What is this K-composition score? Round to three decimals.**

**Task 13. What is the mean K-composition score over all 144 pairs of heads of layers 4 and 5? Round to three decimals.**

## **Conclusion**

- The logit decomposes exactly into direct contributions of blocks if the scale of the final LayerNorm is fixed: the sum of contributions plus $b_U$ equals the model logit.
- The contribution of an attention block decomposes further by heads, and that of an MLP layer by neurons: the activation of a neuron is multiplied by the projection of its value vector onto the unembedding column.
- The value vector of a neuron projected onto the vocabulary shows which tokens the neuron promotes on any input; this is reading the MLP as key-value memory.
- Composition scores are computed from the weights, without data, and show which pairs of heads can read each other's outputs. Whether a pair actually works together on text is checked by intervention: these are the topics of the section "Circuits".